In [1]:
import pandas as pd 
import duckdb
import os
import requests
from datetime import datetime
from dotenv import load_dotenv , find_dotenv
import math 

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
OPINET_API_KEY_GROUP = ['OPINET_API_KEY_1', 'OPINET_API_KEY_2', 'OPINET_API_KEY_3', 
                        'OPINET_API_KEY_4', 'OPINET_API_KEY_5', 'OPINET_API_KEY_6' ] 
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT') 
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')


#2. duckdb를 통한 s3 읽기 설정
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET s3_endpoint='{MINIO_ENDPOINT}';")
con.execute(f"SET s3_access_key_id='{MINIO_ACCESS_KEY}';")
con.execute(f"SET s3_secret_access_key='{MINIO_SECRET_KEY}';")
con.execute("SET s3_url_style='path'; SET s3_use_ssl='false';")


print("✅ DuckDB의 MinIO 접속 준비 완료!")



✅ DuckDB의 MinIO 접속 준비 완료!


In [ ]:
# A1. 기초적인 Pearson Correlation 계산
df = con.sql(
    """
    with base as (
    select 
    part_dt
    , max(case when prodcd = 'B027' then price end ) as gasoline_prc
    , max(case when prodcd = 'D047' then price end ) as disel_prc
    from read_parquet('s3://petroleum-project/national_avg/price/*/*.parquet')
    where 1=1
    and part_dt >= '20260101'
    group by 1
    )
    select 
    CORR(gasoline_prc , disel_prc) as pearson_correlation
    from base 
"""
).df()

display(df)

,pearson_correlation
0,0.99153


In [ ]:
# A2. 연도별 Pearson Correlation 계산
import matplotlib

df = con.sql(
    """
    with base as (
    select 
    part_dt
    , max(case when prodcd = 'B027' then price end ) as gasoline_prc
    , max(case when prodcd = 'D047' then price end ) as disel_prc
    from read_parquet('s3://petroleum-project/national_avg/price/*/*.parquet')
    where 1=1
    group by 1
    )
    select 
    substring( cast(part_dt AS STRING), 1,4) as yyyy
    , CORR(gasoline_prc , disel_prc) as pearson_correlation
    from base 
    group by 1
    order by 1
"""
).df()



,yyyy,pearson_correlation
0,2008,0.986442
1,2009,0.871884
2,2010,0.976378
3,2011,0.943852
4,2012,0.958399
5,2013,0.954182
6,2014,0.997692
7,2015,0.838822
8,2016,0.988086
9,2017,0.999460
